# Step 17. Why is cohort A the most discriminating?

Reads `step12_panels.rds`. Same study, same parameters, a random split into 87 / 87 / 86, yet
A yields 12 trait-associated modules, B yields 1 and C yields 4.

This step reports what differs between the cohorts and draws no conclusion beyond the numbers.

In [1]:
source("../src/paths.R")
options(stringsAsFactors = FALSE)
P <- readRDS(art("step12_panels.rds")); D <- P$D; spec <- P$spec; SITES <- P$SITES
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

build_traits <- function(m, spec) {
  out <- data.frame(row.names = rownames(m))
  for (i in seq_len(nrow(spec))) {
    v <- m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i],
      numeric = as.numeric(as.character(v)),
      binary  = as.numeric(v == "Positive"),
      ordinal = { lvl <- unique(v[!is.na(v) & v != ""])
                  lvl <- lvl[order(as.numeric(sub("-.*", "", lvl)))]
                  as.integer(factor(v, levels = lvl, ordered = TRUE)) })
  }
  out
}
TR <- lapply(SITES, function(s) build_traits(W[[s]]$meta, spec)); names(TR) <- SITES

cat("network structure per cohort (all15):\n")
print(do.call(rbind, lapply(SITES, function(s) {
  d <- D[[paste(s, "all15")]]; m <- W[[s]]$mods
  data.frame(cohort = s, n = nrow(W[[s]]$X),
             modules = length(setdiff(unique(m), "grey")), grey = sum(m == "grey"),
             largest = max(table(m[m != "grey"])),
             largest_pct = round(100 * max(table(m[m != "grey"])) / P$PANEL_TOTAL, 1),
             associated = length(d$sig))
})), row.names = FALSE)

cat("\ntrait spread per cohort (sd for numeric/ordinal, positive count for binary):\n")
sp <- do.call(rbind, lapply(spec$name, function(t) {
  row <- data.frame(trait = t, type = spec$type[spec$name == t])
  for (s in SITES) {
    v <- TR[[s]][[t]]
    row[[s]] <- if (spec$type[spec$name == t] == "binary") sum(v == 1, na.rm = TRUE)
                else round(sd(v, na.rm = TRUE), 2)
  }
  row
}))
print(sp, row.names = FALSE)

cat("\nbatch composition (the split was stratified on Batch only):\n")
print(do.call(rbind, lapply(SITES, function(s)
  data.frame(cohort = s, t(as.matrix(table(W[[s]]$meta$Batch)))))), row.names = FALSE)

saveRDS(list(spread = sp), art("step17_diagnostics.rds"))

network structure per cohort (all15):


 cohort  n modules grey largest largest_pct associated
      A 87      52 1016    2216        30.4         12
      B 87      23 1667    2784        38.2          1
      C 86      46 1377    2656        36.4          4



trait spread per cohort (sd for numeric/ordinal, positive count for binary):


            trait    type      A      B      C
        SLEDAI_2K numeric   3.55   2.94   2.78
         C3_level numeric   0.30   0.32   0.29
         C4_level numeric   0.11   0.12   0.11
   Duration_years numeric  10.30  10.95   9.70
 Lymphocyte_count numeric   0.85   0.90   0.54
             uPCR numeric 279.20 151.51 179.89
       Creatinine numeric  44.25  60.42  73.78
        Sm_status  binary  21.00  18.00  24.00
    RNP_68_status  binary  11.00   7.00  18.00
     RNP_A_status  binary  20.00  20.00  28.00
     Ro_52_status  binary  24.00  30.00  25.00
     Ro_60_status  binary  36.00  46.00  38.00
        La_status  binary  11.00   9.00  11.00
     dsDNA_status  binary  26.00  28.00  16.00
         Age_band ordinal   2.89   2.71   2.67



batch composition (the split was stratified on Batch only):


 cohort  A  B
      A 69 18
      B 69 18
      C 69 17


## What the numbers say

**It is not the patients.** Batch composition is identical by construction, 69/18, 69/18, 69/17, and trait spread is close across all three cohorts on every attribute.

**It is the network.** A fragments into 52 modules with 1,016 probes unassigned and a
largest module of 30.4% of the panel. B gives 23 modules, 1,667 unassigned, largest
38.2%. B has fewer and coarser eigengenes, so there is less that is specific enough to
correlate with anything. C sits between them at 46 modules.

**Still open.** `modulePreservation` across A, B and C, the adjusted Rand index (ARI) in step 12 compares partitions
within a cohort, which is not the permutation-based preservation statistic, and whether a freely
recovered module contains the *same proteins* across cohorts, not merely the same count.

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [ ]:
run_provenance()